# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hajergafsi/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Lane: Refresh / Content Opportunity Scoring**

This is fundamentally a **ranking / scoring** task, not plain classification — the real decision is "which pages first," and the output a reviewer uses is an ordered list, not an isolated yes/no per page.

Under the hood, I'll get there via a **classification model**: train a model to predict a binary future outcome (declines / doesn't decline), then use its predicted probability as the ranking score. So: classification model, ranking use-case. This mirrors how the starter pipeline works (logistic regression / tree / random forest probabilities feed a final ranked score) — but my label and windowing will be my own (see Section 2).

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Starter's label (for reference):** `trend_direction == "down"` — a bucket computed from the *current* 90-day window. This is a proxy, not a future outcome, and the framing skill flags this pattern directly: "a label that comes from someone's rule means your model learns the rule, not the world." I am not using this as my capstone target — only as a reference point.

**My capstone target — a genuine future-OBSERVED outcome:**

> Using the prior 90 days of features → did the page's search visibility decline over the next 30 days?

Concretely (final thresholds to be set with real data in ML-04):
- Feature window: `report_date` in `[T-90, T]`
- Target window: `report_date` in `[T, T+30]`
- Label = 1 if impressions/clicks drop by more than X% from the feature window's trailing rate to the target window's rate, sustained (not a single-day blip), AND the page had enough volume in the feature window to be meaningful (minimum-impressions threshold, set in ML-04/07).
- Label = 0 otherwise.

**Two corrections that must be baked into this label before it's real, not just theoretical:**

1. **Per-client window anchoring.** Client history depth varies wildly (`dim_clients.gsc_data_start` / `ga4_data_start`). I cannot slap one global calendar window (e.g. "Jan 2026 → Feb 2026") across all clients — a client whose tracking only started recently would show a fake "decline" that's actually just "no data yet." Every `T-90/T/T+30` window must be anchored relative to *that client's own* tracking start, not a fixed calendar date.
2. **GA4 zero-fill filter.** Rows before a client's `ga4_data_start` have GA4 columns zero-filled with `ga4_data_available = FALSE`. If my label or features ever use a GA4-sourced signal (sessions, engagement), I must filter on that flag first — otherwise "not tracked yet" silently becomes "zero engagement," which would corrupt the label.

This is an OBSERVED outcome (something that actually happened after the fact), not a rule defined from the same window as my features — that's the core upgrade over the starter, and the whole point of this section per the framing skill's rule: *the target must be observed, not defined.*

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary metric: Precision@50.**

Why: reviewers can only act on a limited number of flagged pages per cycle (I'll treat 50 as a stand-in for "one reviewer's weekly capacity" — a threshold choice I'll defend with real numbers in ML-07). Precision@50 answers exactly the question that matters for this decision: *of the top 50 pages I tell someone to check, how many actually did decline?*

**Secondary metrics I'll also report** (context, not the primary bar):
- Average precision (quality of the whole ranking, not just the top 50)
- ROC-AUC (a standard comparison point, useful context against the starter's published numbers — though not a direct "beat this" target, since the starter used a different label, window, and data slice)

**"Good" means:** my trained model's Precision@50 clearly beats my *own* rule-based baseline (built in ML-07), evaluated on my *own* future-window label — not a comparison to the starter's reported 12/50 or 37/50.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Shape:", df.shape)
print("content_id uniqueness check:", df["content_id"].is_unique)
print("client_id count:", df["client_id"].nunique())

df.head(3)

**One row (starter CSV) = one content item's trailing-90-day performance snapshot**, pseudonymized, one per `content_id`.

**Important limitation:** this starter CSV's grain (one static 90-day snapshot per page) *cannot* actually support my future-window label as defined above — there's no `report_date` column here to split into a feature window and a target window inside a single pre-aggregated snapshot. This is exactly why my capstone will move to the warehouse's `fact_content_daily_performance` (or its `fact_content_daily_performance_sample`, ~11.7M rows — I'll iterate on the sample and only run the full ~79M-row table once my query is final, to avoid HTTP 429 rate limits). That table's grain is `report_date × client_hash_id × content_hash_id`, with `gsc_avg_position` as the position column (note: **not** `avg_position` — that name is specific to the starter CSV).

I'm using the starter CSV here only to sanity-check the unit of analysis and basic distributions before building the real windowed dataset in ML-04/05.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule *can* approximate this — that's exactly what my baseline in ML-07 will be (e.g. "flag pages with an X% impression drop in the last 30 days of the feature window, if volume ≥ threshold"). But a single rule struggles because decline risk plausibly depends on several signals moving together in ways that are hard to hand-encode:

- a mild visibility drop combined with a worsening position trend
- stable impressions but falling CTR (attention loss even without a visibility drop)
- freshness/age interacting differently across content types
- normal seasonal dips that look like decline under a simple threshold rule but aren't

A model can weigh and combine these signals, learn nonlinear interactions, and output calibrated probabilities — a fixed if-statement can encode maybe 2–3 rules before it becomes unmanageable and arbitrary. My job in ML-08 is to actually *prove* the model earns this added complexity, by testing it against my own baseline on my own label — not just assuming "ML is better."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.